In [1]:
%load_ext autoreload
%reload_ext autoreload

import logging
import os
import time
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from dataclasses import asdict

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from shapely import wkt

import utils.route_utils as ru
from config.config import (
  FILTER_OPTIONS,
  FITNESS_WEIGHTS,
  N_NETWORKS,
  N_ROUTES,
  STOP_MIN,
  WALKING_DISTANCES,
  WD_TO_USE,
)
from genetic_algorithm.crossover import crossover
from genetic_algorithm.fitness import fitness_score, make_run_tag
from genetic_algorithm.mutate import mutate
from genetic_algorithm.schema import HyperParameters, Network, RunState
from utils.checkpoint_utils import load_data, load_ga_checkpoint, save_ga_checkpoint
from utils.graph_utils import load_graph, save_route_graph_to_plot
from utils.time_utils import ElapsedTimeFormatter, format_elapsed


In [2]:
# Logger object. Use debug, info, warning, error accordingly
logger = logging.getLogger("genetic_algorithm")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
  handler = logging.StreamHandler()

  formatter = ElapsedTimeFormatter("[%(elapsed)s] %(levelname)s: %(message)s")

  handler.setFormatter(formatter)
  logger.addHandler(handler)

logger.propagate = False

In [3]:
# GA VERSION 3 by sir Anthony and friends
def nsga_ii(
  initial_population: list[Network], graph_of_stops: nx.MultiDiGraph, allowed_edge: dict, edge_len: dict, total_network_road_distance: float,
  n_transport_nodes: int, hyperparameters: dict[str, any], city_name: str, population_gdf: gpd.GeoDataFrame, executor: ThreadPoolExecutor | ProcessPoolExecutor | None = None,
  run_tag = "", checkpoint = None, debug: bool = False
):
  # Sets logging to debug mode meaning logger.debug() works
  logger.setLevel(logging.DEBUG if debug else logging.INFO)
  hp: HyperParameters = (
    hyperparameters
    if isinstance(hyperparameters, HyperParameters)
    else HyperParameters.from_dict(hyperparameters)
  )

  logger.info(f"=== GA Configuration | City: {city_name} | Pop Size: {len(initial_population)} ===")
  logger.info("Execution Mode : " + "Sequential (single-threaded)" if not executor else f"Threaded ({type(executor).__name__})")
  logger.info("Hyperparameters:")
  for k, v in asdict(hp).items():
    logger.info(f" └─ {k:<28}: {v}")

  if checkpoint is None:
    state = RunState(network_population=initial_population)
    logger.info("No checkpoint found. Starting fresh GA run...")
  else:
    state = RunState.from_checkpoint(checkpoint)
    logger.info(
      "Resumed at generation %d (%.1fs accumulated).",
      state.generation_num,
      state.elapsed_before,
    )

  started = time.monotonic()
  stage_times = defaultdict(float)

  def save():
    logger.info(f"Saving at generation {state.generation_num} ({elapsed():.2f}s)...")
    save_ga_checkpoint(f"_{run_tag}", state.to_checkpoint(elapsed()), city_name)

  def elapsed() -> float:
    return state.elapsed_before + (time.monotonic() - started)

  def evaluate_network(network):
    network.fitness_score, network.fitness_score_components = fitness_score(
      network,
      hp.fitness_weights,
      n_transport_nodes,
      total_network_road_distance,
      allowed_edge,
      edge_len,
      graph_of_stops,
      population_gdf,
    )

  try:
    while state.generation_num <= hp.max_generations:
      gen_start = time.perf_counter()
      generation_num = state.generation_num

      # STAGE 0: Evaluate the fitness of each network in the population
      t0 = time.perf_counter()
      unevaluated = [n for n in state.network_population if n.fitness_score is None]

      if executor:
        if unevaluated:
          list(executor.map(evaluate_network, unevaluated))
      else:
        for network in unevaluated:
          network.fitness_score, network.fitness_score_components = fitness_score(network, hp.fitness_weights, n_transport_nodes, total_network_road_distance, allowed_edge, edge_len, graph_of_stops, population_gdf)
      t_init_eval = time.perf_counter() - t0
      stage_times["initial_eval"] += t_init_eval

      # Sort the network population by fitness score
      sorted_network_population = sorted(state.network_population, key=lambda x: x.fitness_score, reverse=True)
      fitness_scores = np.array([network.fitness_score for network in sorted_network_population])

      max_score = sorted_network_population[0].fitness_score
      ave_score = np.average(fitness_scores)
      logger.info(f"Generation {generation_num} Max Score: {max_score:.4f} Average Score: {ave_score:.4f}")

      # The max network score of this generation
      state.max_score_list.append([max_score, sorted_network_population[0].fitness_score_components])

      # Check if this generation has the highest max fitness score and highest average score
      if max_score >= state.best_max_score and ave_score >= state.best_avg_score:
        state.best_max_score, state.best_avg_score = max_score, ave_score
        state.best_population = sorted_network_population

      # Check if score plateaus based on max score
      if (
        state.prev_max_score is not None
        and abs(max_score - state.prev_max_score) <= hp.plateau_threshold
      ):
        state.plateau_counter = state.plateau_counter + 1
        if state.plateau_counter >= hp.plateau_count_limit:
          logger.info(f"Plateau detected in generation {generation_num}. Stopping...")
          break
      else:
        state.plateau_counter = 0
      state.prev_max_score = max_score

      # STAGE 1: Selecting 10% of the networks to be parents using Roulette Wheel Selection
      t0 = time.perf_counter()

      # Getting the number of parents to be selected
      num_parents = int(len(state.network_population) * hp.selection_rate)
      num_parents = max(2, num_parents)
      if num_parents % 2 == 1:
        num_parents += 1

      logger.debug(f"Number of parents selected: {num_parents}")

      # Get the probabilities
      weights = fitness_scores - np.min(fitness_scores)
      total_score = np.sum(weights)
      if total_score > 0:
        selection_p = weights / total_score
        selection_p /= np.sum(selection_p)
      else:
        selection_p = np.full(len(weights), 1.0 / len(weights))

      parent_networks = np.random.choice(sorted_network_population, num_parents, replace=False, p=selection_p).tolist()

      # Pairing the parents
      parent_pairs = [(parent_networks[i], parent_networks[i + 1]) for i in range(0, len(parent_networks), 2)]
      t_sel = time.perf_counter() - t0
      stage_times["selection"] += t_sel

      # STAGE 2: Genetic Operators via. Crossover and Mutation
      t_crossover, t_mutation = 0.0, 0.0
      offspring_networks = []
      for parent1, parent2 in parent_pairs:
        tc0 = time.perf_counter()
        # Get 2 offspring from crossovers between the two parents
        child1, child2, is_different = crossover(parent1, parent2, graph_of_stops)
        t_crossover += time.perf_counter() - tc0

        tm0 = time.perf_counter()
        # Getting the number of mutations. If the offspring are entirely different from their parents, there should be atleast one mutation
        table = (hp.mutations_per_child if is_different else hp.mutations_per_child_identical)
        num_mutations = int(np.random.choice(len(table), p=table))

        # Apply mutations to the offspring based on mutation probability hyperparameter
        for child in (child1, child2):
          for _ in range(num_mutations):
            if np.random.rand() < hp.mutation_probability:
              mutate(child, graph_of_stops, hp.walk_distance)
        t_mutation += time.perf_counter() - tm0

        # Add the offspring to the list of offspring
        offspring_networks.extend((child1, child2))

      stage_times["crossover"] += t_crossover
      stage_times["mutation"] += t_mutation

      # STAGE 3: Elitism
      t0 = time.perf_counter()

      # Evaluate offspring networks for elitism
      if executor:
        if offspring_networks:
          list(executor.map(evaluate_network, offspring_networks))
      else:
        for offspring in offspring_networks:
          offspring.fitness_score, offspring.fitness_score_components = fitness_score(offspring, hp.fitness_weights, n_transport_nodes, total_network_road_distance, allowed_edge, edge_len, graph_of_stops, population_gdf)

      # Elitism: combining the current and offspring population into a single pool 2N, then selecting the best N individuals
      if offspring_networks:
        pop_size = len(state.network_population)
        num_elites = (max(1, int(pop_size * hp.elitism_rate)) if hp.elitism_rate > 0 else 0)

        elites = sorted_network_population[:num_elites]

        # Fill the remaining slots from offspring + current pop
        candidate_pool = sorted(offspring_networks + sorted_network_population[num_elites:], key=lambda x: x.fitness_score, reverse=True)
        state.network_population = elites + candidate_pool[:pop_size - num_elites]

      t_elitism = time.perf_counter() - t0
      stage_times["elitism"] += t_elitism

      gen_total = time.perf_counter() - gen_start
      logger.debug(
        f"Gen {generation_num} ({gen_total:.3f}s) | "
        f"init_eval={t_init_eval:.3f}s, sel={t_sel:.4f}s, "
        f"xover={t_crossover:.3f}s, mut={t_mutation:.3f}s, "
        f"elitism={t_elitism:.4f}s"
      )

      # Increment the generation number
      state.generation_num += 1

      if state.generation_num % hp.checkpoint_every == 0:
        save()

      logger.debug(f"Completed generation {generation_num}. Proceeding to next generation...")

  except KeyboardInterrupt:
    save()

  runtime = elapsed()
  logger.info(f"Highest score of all generations: {state.best_max_score}")
  logger.info(f"Total GA runtime: {format_elapsed(runtime)}")

  total_tracked = sum(stage_times.values()) or 1.0
  logger.info("Runtime Breakdown by Stage:")
  for stage, secs in sorted(stage_times.items(), key=lambda x: x[1], reverse=True):
    pct = (secs / total_tracked) * 100
    logger.info(f" └─ {stage:<16}: {secs:8.2f}s ({pct:5.1f}%)")

  return state.best_population or [], state.max_score_list

In [4]:
HYPERPARAMETERS = {
  "fitness_weights": FITNESS_WEIGHTS,
  "elitism_rate": 0.1,
  "selection_rate": 0.4,
  "mutation_probability": 0.35,
  "mutations_per_child": (0.25, 0.45, 0.20, 0.10),
  "mutations_per_child_identical": (0.0, 0.40, 0.40, 0.20),
  "plateau_threshold": 1e-5,
  "plateau_count_limit": 200,
  "max_generations": 1000,
  "checkpoint_every": 50,
  "walk_distance": WALKING_DISTANCES[WD_TO_USE],
}

In [5]:
CITY_NAME = "Makati"  # Taguig, Mandaluyong, Manila
INITIAL_RS_DIR = f"../data/initial_route_sets/{CITY_NAME}"

irsg_run_tag = ru.make_run_tag(WALKING_DISTANCES[WD_TO_USE], STOP_MIN, N_ROUTES[CITY_NAME], N_NETWORKS)
file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_population{irsg_run_tag}.pkl")
initial_population = load_data(file_path)

file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_stops{irsg_run_tag}.pkl")
graph_of_stops = load_data(file_path)

graph_dir = os.path.join("../data/street_networks", f"{CITY_NAME}.graphml")
city_graph = load_graph(graph_dir)

filepath = f'../data/population_distribution/{CITY_NAME.lower()}-population-polygon.csv'
df = pd.read_csv(filepath)
df['geometry'] = df['geometry'].apply(wkt.loads)
population_gdf = gpd.GeoDataFrame(df, crs='epsg:32651')

Graph loaded successfully
NUMBER OF EDGES:  5558
NUMBER OF NODES:  2294


In [6]:
def validate_network(network: Network, master_graph: nx.MultiDiGraph) -> None:
  for route_obj in network.routes:
    for conn in route_obj.route:
      u, v = conn[0], conn[-1]
      assert u in master_graph, (
        f"Stop {u} in {route_obj.route_id} missing from master_graph"
      )
      assert v in master_graph, (
        f"Stop {v} in {route_obj.route_id} missing from master_graph"
      )
      assert u in network.graph, f"Stop {u} missing from network.graph"
      assert v in network.graph, f"Stop {v} missing from network.graph"
      edge = network.graph.get_edge_data(u, v, key=route_obj.route_id)
      assert edge is not None and "distance" in edge, (
        f"Missing edge/distance ({u}, {v}, {route_obj.route_id})"
      )

for net in initial_population:
  validate_network(net, graph_of_stops)

print("IRS validated.")

IRS validated.


In [7]:
n_transport_nodes = sum(
  1 for _, d in graph_of_stops.nodes(data=True) if d.get("isTranspo")
)

allowed_edge = {}
edge_len = {}
total_network_road_distance = 0.0

ga_run_tag = make_run_tag(HYPERPARAMETERS["fitness_weights"])

resume = False  # Set to true if u wanna use a checkpoint
checkpoint = load_ga_checkpoint(f"_{ga_run_tag}", CITY_NAME) if resume else None

for u, v, k, data in city_graph.edges(keys=True, data=True):
  if data.get("junction") is not None:
    continue
  hw = data.get("highway")
  allowed = False
  if isinstance(hw, list):
    allowed = any(h in FILTER_OPTIONS for h in hw)
  else:
    allowed = hw in FILTER_OPTIONS

  key = frozenset((u, v))
  allowed_edge[key] = allowed
  edge_len[key] = data.get("length", 0.0)
  if isinstance(hw, str) and allowed:
    total_network_road_distance += edge_len[key]

cpu_count = os.cpu_count()
assert cpu_count is not None
# N_WORKERS = 50
# ctx = get_context("fork")
# with ProcessPoolExecutor(max_workers=min(cpu_count, N_WORKERS), mp_context=ctx) as executor:
with ThreadPoolExecutor(max_workers=cpu_count) as executor:
  best_network, _ = nsga_ii(
    initial_population,
    graph_of_stops,
    allowed_edge,
    edge_len,
    total_network_road_distance,
    n_transport_nodes,
    HYPERPARAMETERS,
    CITY_NAME,
    population_gdf=population_gdf,
    executor=executor,
    run_tag=ga_run_tag,
    checkpoint=checkpoint,
    debug=True,
  )

 0s] INFO: === GA Configuration | City: Makati | Pop Size: 100 ===
 0s] INFO: Threaded (ThreadPoolExecutor)
 0s] INFO: Hyperparameters:
 0s] INFO:  └─ fitness_weights             : {'w_cov': 0.2, 'w_int': 0.1, 'w_ext': 0.1, 'w_vul': 0.2, 'w_rob': 0.2, 'w_red': 0.2}
 0s] INFO:  └─ walk_distance               : 750
 0s] INFO:  └─ elitism_rate                : 0.1
 0s] INFO:  └─ selection_rate              : 0.4
 0s] INFO:  └─ mutation_probability        : 0.35
 0s] INFO:  └─ mutations_per_child         : (0.25, 0.45, 0.2, 0.1)
 0s] INFO:  └─ mutations_per_child_identical: (0.0, 0.4, 0.4, 0.2)
 0s] INFO:  └─ plateau_threshold           : 1e-05
 0s] INFO:  └─ plateau_count_limit         : 200
 0s] INFO:  └─ max_generations             : 1000
 0s] INFO:  └─ checkpoint_every            : 50
 0s] INFO: No checkpoint found. Starting fresh GA run...


ValueError: too many values to unpack (expected 2, got 3)

In [ ]:
# save_route_graph_to_plot(graph_of_stops, best_network[0].routes, "best_network.html", f"../plots/Result Route Networks/{CITY_NAME}")